## QAQC global (Campo + VSOL)

RASCUNHO -- ainda nao testado em Databricks real.

Roda **depois** do `04_envio_sharepoint` nas duas Jobs (Campo e VSOL) -- ver `docs/vsol-integracao.md`.
Nao tem widget `fonte`: le a execucao mais recente de **cada** fonte em `SILVER/api_validado/` (Campo e
VSOL, quando existirem) e junta as duas antes de rodar as checagens. Isso garante que o QAQC sempre usa
o dado mais atual disponivel, não importa qual das duas Jobs disparou esta execucao.

Substitui o notebook solto `04) QAQC.ipynb` (raiz do repo) -- mesma logica de QAQC, reorganizada pra
rodar de forma config-driven (widget `projeto`, sem caminho fixo de workspace pessoal) e gravar em
Delta Lake + SharePoint, igual ao resto do pipeline.

## Setup

In [ ]:
dbutils.library.restartPython()
!pip install --upgrade pip
!pip install openpyxl
!pip install python-dotenv
dbutils.library.restartPython()

In [ ]:
import re
import datetime as _dt
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, when, regexp_replace, regexp_extract, trim, array, expr,
    abs as sabs, first, round as sround,
)

from config import get_config
from sharepoint_connector import upload_file

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao - {_log_err}")

## Projeto

In [ ]:
dbutils.widgets.text("projeto", "")
projeto = dbutils.widgets.get("projeto").strip()
cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

project_path = "/mnt/wst/" + projeto
folder_silver = project_path + "/SILVER/"
api_validado_path = folder_silver + "api_validado/"

## Descoberta das execucoes mais recentes (Campo + VSOL)

`SILVER/api_validado/` ja tem uma subpasta por execucao, nomeada pelo `output_filename` de cada
notebook `01` -- `apiCAMPO_...` pra Campo, `vsolSITE_...` pra VSOL. As duas fontes ja convivem nessa
pasta sem colidir (nomes diferentes por natureza). Aqui a gente pega só a mais recente de cada
prefixo e junta as duas.

In [ ]:
PREFIXOS_FONTE = {"apiCAMPO": "api", "vsolSITE": "vsol"}

try:
    pastas = [f.name.rstrip("/") for f in dbutils.fs.ls(api_validado_path) if f.isDir()]
except Exception:
    pastas = []

dfs_por_fonte = {}
for prefixo, nome_fonte in PREFIXOS_FONTE.items():
    candidatas = sorted([p for p in pastas if p.startswith(prefixo)], reverse=True)
    if not candidatas:
        continue
    mais_recente = candidatas[0]
    try:
        dfs_por_fonte[nome_fonte] = spark.read.format("delta").load(api_validado_path + mais_recente)
    except Exception as _e:
        execucao_steps.append({
            "etapa": f"Leitura api_validado ({nome_fonte})",
            "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
            "observacoes": f"Falha ao ler {mais_recente}: {_e}",
        })

if not dfs_por_fonte:
    execucao_steps.append({
        "etapa": "Descoberta de execucoes (api_validado)",
        "status": "Aviso",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
        "observacoes": f"Nenhuma execucao encontrada em {api_validado_path} -- QAQC nao tem o que processar",
    })
    persistir_log()
    dbutils.notebook.exit("Nenhuma execucao de Campo ou VSOL encontrada em api_validado")

df_api = None
for nome_fonte, df_fonte in dfs_por_fonte.items():
    df_fonte = df_fonte.withColumn("fonte_qaqc", F.lit(nome_fonte))
    df_api = df_fonte if df_api is None else df_api.unionByName(df_fonte, allowMissingColumns=True)

execucao_steps.append({
    "etapa": "Descoberta de execucoes (api_validado)",
    "status": "Sucesso",
    "registros_lidos": df_api.count(),
    "registros_escritos": df_api.count(),
    "flags": "\u2014",
    "observacoes": f"Fontes combinadas: {list(dfs_por_fonte.keys())}",
})

## Surrogate

Lista fixa de compostos surrogate (confirmada com o Sinderley em 2026-09-14) -- checa se o resultado
está fora da faixa 80-120%. Não usa a unidade (`%`) como critério porque parâmetros físicos da amostra
(ex.: Teor de umidade, % de sólidos) também vêm em `%` e não são surrogate.

In [ ]:
PARAMETROS_SURROGATE = [
    "Teor de Dibromofluorometano (%)",
    "Teor de Nitrobenzeno-d5 (%)",
    "Teor de Terfenil-d14 (%)",
    "Teor de 2-Fluorobifenil (%)",
    "Teor de 2.4.6-Tribromofenol (%)",
    "Teor de 4-Bromofluorbenzeno (VOC) (%)",
    "Teor de 1.2-Dicloroetano (%)",
    "Teor de Tolueno-d8 (%)",
    "2.6-Dibromofenol (%)",
]

df_api = df_api.withColumn(
    "flag_surrogate",
    when(
        trim(col("chemical_name")).isin(PARAMETROS_SURROGATE) & ~col("value").between(80, 120),
        "Resultado fora da faixa de 80 a 120"
    ).otherwise(None)
)

qtd_surrogate_flag = df_api.filter(col("flag_surrogate").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - Surrogate",
    "status": "Sucesso" if qtd_surrogate_flag == 0 else "Aviso",
    "registros_lidos": df_api.filter(trim(col("chemical_name")).isin(PARAMETROS_SURROGATE)).count(),
    "registros_escritos": qtd_surrogate_flag,
    "flags": _collect_flags(df_api, ["flag_surrogate"]),
    "observacoes": (
        "Todos os surrogates dentro da faixa 80-120%" if qtd_surrogate_flag == 0
        else f"{qtd_surrogate_flag} resultado(s) de surrogate fora da faixa 80-120%"
    ),
})

## Total vs. Dissolvido

Portado de `04) QAQC.ipynb` -- mesma regra, agora com log de execucao.

In [ ]:
AMOSTRAS_EXCLUIR_QAQC = cfg.get("qaqc_amostras_excluir", [])

df_api = df_api.withColumn(
    "parametro_base",
    trim(regexp_replace(col("chemical_name"), "(?i)(total|dissolvido)", ""))
)

df_matriz_filtrada = df_api.filter(
    col("samp_matrix").isin("Agua Subterranea", "Fase livre")
).filter(
    ~col("sample_name").isin(AMOSTRAS_EXCLUIR_QAQC)
).filter(
    ~col("chemical_name").rlike("(?i)cromo")
)

df_totais = df_matriz_filtrada.filter(col("chemical_name").rlike("(?i)total")) \
    .select(col("sample_name"), col("parametro_base"), col("value").alias("valor_total"))

df_dissolvidos = df_matriz_filtrada.filter(col("chemical_name").rlike("(?i)dissolvido")) \
    .select(col("sample_name"), col("parametro_base"), col("value").alias("valor_dissolvido"))

df_comparacao = df_totais.join(
    df_dissolvidos, on=["sample_name", "parametro_base"], how="inner"
).withColumn(
    "flag_totais_dissolvido",
    when(col("valor_dissolvido") > col("valor_total"), "Dissolvido maior que Total").otherwise(None)
).select("sample_name", "parametro_base", "valor_total", "valor_dissolvido", "flag_totais_dissolvido")

df_api = df_api.join(
    df_comparacao, on=["sample_name", "parametro_base"], how="left"
).drop("valor_dissolvido", "valor_total", "parametro_base")

qtd_td_flag = df_api.filter(col("flag_totais_dissolvido").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - Total vs Dissolvido",
    "status": "Sucesso" if qtd_td_flag == 0 else "Aviso",
    "registros_lidos": df_comparacao.count(),
    "registros_escritos": qtd_td_flag,
    "flags": _collect_flags(df_api, ["flag_totais_dissolvido"]),
    "observacoes": (
        "Nenhum par Total/Dissolvido inconsistente" if qtd_td_flag == 0
        else f"{qtd_td_flag} par(es) com Dissolvido maior que Total"
    ),
})

## DBO vs. DQO

Portado de `04) QAQC.ipynb` -- mesma regra, agora com log de execucao.

In [ ]:
df_matriz_filtrada = df_api.filter(
    col("samp_matrix").isin("Agua Subterranea", "Fase livre")
)

df_dbo = df_matriz_filtrada.filter(col("chemical_name").rlike("(?i)^DBO")) \
    .select(col("sample_name"), col("value").alias("valor_dbo"))

df_dqo = df_matriz_filtrada.filter(col("chemical_name").rlike("(?i)^DQO")) \
    .select(col("sample_name"), col("value").alias("valor_dqo"))

df_comparacao_dbo_dqo = df_dbo.join(
    df_dqo, on=["sample_name"], how="inner"
).withColumn(
    "flag_dbo_dqo",
    when(col("valor_dbo") > col("valor_dqo"), "DBO maior que DQO").otherwise(None)
).select("sample_name", "valor_dbo", "valor_dqo", "flag_dbo_dqo")

df_api = df_api.join(df_comparacao_dbo_dqo, on=["sample_name"], how="left")

qtd_dbo_dqo_flag = df_api.filter(col("flag_dbo_dqo").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - DBO vs DQO",
    "status": "Sucesso" if qtd_dbo_dqo_flag == 0 else "Aviso",
    "registros_lidos": df_comparacao_dbo_dqo.count(),
    "registros_escritos": qtd_dbo_dqo_flag,
    "flags": _collect_flags(df_api, ["flag_dbo_dqo"]),
    "observacoes": (
        "Nenhuma amostra com DBO maior que DQO" if qtd_dbo_dqo_flag == 0
        else f"{qtd_dbo_dqo_flag} amostra(s) com DBO maior que DQO"
    ),
})

## Cromo

Portado de `04) QAQC.ipynb` -- mesma regra (4 checagens entre as especies), agora com log de execucao.

In [ ]:
NOMES_CROMO = [
    "Cromo Dissolvido (mg/L)",
    "Cromo Hexav Dissolvido (mg/L)",
    "Cromo Hexavalente (mg/L)",
    "Cromo Total (mg/L)",
]

df_cromo = df_api.filter(
    col("samp_matrix").isin("Agua Subterranea", "Fase livre")
).filter(
    ~col("sample_name").isin(AMOSTRAS_EXCLUIR_QAQC)
).filter(
    trim(col("chemical_name")).isin(NOMES_CROMO)
)

df_cromo_pivot = df_cromo.groupBy("sample_name").pivot("chemical_name", NOMES_CROMO).agg({"value": "first"})
df_cromo_pivot = df_cromo_pivot.toDF(
    "sample_name", "valor_dissolvido", "valor_hexav_dissolvido", "valor_hexavalente", "valor_total"
)

df_cromo_pivot = df_cromo_pivot.withColumn(
    "flags_cromo",
    array(
        when(col("valor_hexav_dissolvido") > col("valor_hexavalente"), lit("Hexav Dissolvido > Hexavalente")),
        when(col("valor_dissolvido") > col("valor_total"), lit("Dissolvido > Total")),
        when(col("valor_hexav_dissolvido") > col("valor_dissolvido"), lit("Hexav Dissolvido > Dissolvido")),
        when(col("valor_hexavalente") > col("valor_total"), lit("Hexavalente > Total")),
    )
)
df_cromo_pivot = df_cromo_pivot.withColumn(
    "flag_cromo",
    expr("nullif(concat_ws('; ', filter(flags_cromo, x -> x is not null)), '')")
).select("sample_name", "flag_cromo")

df_api = df_api.join(df_cromo_pivot, on="sample_name", how="left")
df_api = df_api.withColumn(
    "flag_cromo",
    when(trim(col("chemical_name")).isin(NOMES_CROMO), col("flag_cromo")).otherwise(None)
)

qtd_cromo_flag = df_api.filter(col("flag_cromo").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - Cromo",
    "status": "Sucesso" if qtd_cromo_flag == 0 else "Aviso",
    "registros_lidos": df_cromo_pivot.count(),
    "registros_escritos": qtd_cromo_flag,
    "flags": _collect_flags(df_api, ["flag_cromo"]),
    "observacoes": (
        "Nenhuma inconsistencia entre as especies de cromo" if qtd_cromo_flag == 0
        else f"{qtd_cromo_flag} linha(s) de cromo com inconsistencia"
    ),
})

## Branco (QC blank)

Portado de `04) QAQC.ipynb` -- agora com log de execucao.

In [ ]:
df_api = df_api.withColumn(
    "flag_branco",
    when(
        col("quality_code").isin("BC", "BV", "BE") & (col("value") > col("quantification_limit")),
        "Acima_LQ"
    ).otherwise(None)
)

qtd_branco_flag = df_api.filter(col("flag_branco").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - Branco",
    "status": "Sucesso" if qtd_branco_flag == 0 else "Aviso",
    "registros_lidos": df_api.filter(col("quality_code").isin("BC", "BV", "BE")).count(),
    "registros_escritos": qtd_branco_flag,
    "flags": _collect_flags(df_api, ["flag_branco"]),
    "observacoes": (
        "Nenhum branco acima do limite de quantificacao" if qtd_branco_flag == 0
        else f"{qtd_branco_flag} branco(s) acima do limite de quantificacao"
    ),
})

## RPD / Replica -- PENDENTE

Deixado exatamente como estava em `04) QAQC.ipynb`: par de amostra original/réplica **hardcoded** por
nome. O Sinderley ainda vai definir como a coluna `parent` vai identificar réplica/original (2026-09-14)
-- quando isso existir, trocar esse bloco por uma versão que descobre os pares automaticamente via
`parent`, em vez dos nomes fixos abaixo. Até lá, só funciona pro par especifico do BA listado aqui.

In [ ]:
# TODO: substituir por pareamento automatico via coluna `parent` assim que ela existir no schema.
AMOSTRA_ORIGINAL_RPD = "1233-AA-BA-07-ASUB-310726"
AMOSTRA_REPLICA_RPD = "PM-122-ASUB-310726"

df_par = df_api.filter(col("sample_name").isin(AMOSTRA_ORIGINAL_RPD, AMOSTRA_REPLICA_RPD))

df_rpd = df_par.groupBy("chemical_name").pivot(
    "sample_name", [AMOSTRA_ORIGINAL_RPD, AMOSTRA_REPLICA_RPD]
).agg(first("value")) \
 .withColumnRenamed(AMOSTRA_ORIGINAL_RPD, "value_original") \
 .withColumnRenamed(AMOSTRA_REPLICA_RPD, "value_replica")

df_rpd = df_rpd.withColumn(
    "rpd",
    when(
        (col("value_original") == 0) & (col("value_replica") == 0), 0.0
    ).otherwise(
        sabs(col("value_original") - col("value_replica")) /
        ((col("value_original") + col("value_replica")) / 2) * 100
    )
).withColumn(
    "rpd", sround(col("rpd"), 2)
).withColumn(
    "flag_rpd",
    when(col("rpd") > 20, "RPD > 20%").otherwise(None)
).select("chemical_name", "flag_rpd")

df_api = df_api.join(df_rpd, on="chemical_name", how="left")
df_api = df_api.withColumn(
    "flag_rpd",
    when(col("sample_name").isin(AMOSTRA_ORIGINAL_RPD, AMOSTRA_REPLICA_RPD), col("flag_rpd")).otherwise(None)
)

qtd_rpd_flag = df_api.filter(col("flag_rpd").isNotNull()).count()
execucao_steps.append({
    "etapa": "QAQC - RPD/Replica",
    "status": "Sucesso" if qtd_rpd_flag == 0 else "Aviso",
    "registros_lidos": df_par.count(),
    "registros_escritos": qtd_rpd_flag,
    "flags": _collect_flags(df_api, ["flag_rpd"]),
    "observacoes": (
        "RPD dentro do limite (par hardcoded, pendente de generalizacao)" if qtd_rpd_flag == 0
        else f"{qtd_rpd_flag} parametro(s) com RPD acima de 20% (par hardcoded, pendente de generalizacao)"
    ),
})

## Exportacao consolidada (um Excel so, todas as checagens)

Cada aba traz só as linhas flagadas daquela checagem -- igual ao padrão de `export_excel_multisheet`
do `03_validacoes.ipynb`.

In [ ]:
import os

def export_qaqc_excel(sheets: dict, folder_path: str):
    non_empty = {name: df for name, df in sheets.items() if not df.rdd.isEmpty()}

    if not non_empty:
        execucao_steps.append({
            "etapa": "Exportacao QAQC (Excel)",
            "status": "Aviso",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
            "observacoes": "Nenhuma checagem com linha flagada -- Excel nao gerado",
        })
        return None

    data_str = _dt.datetime.now().strftime("%Y%m%d")
    filename = f"qaqc_{data_str}.xlsx"
    dbfs_path = f"{folder_path}/{filename}"
    local_tmp_path = f"/tmp/{filename}"

    if os.path.exists(local_tmp_path):
        os.remove(local_tmp_path)

    total_linhas = 0
    with pd.ExcelWriter(local_tmp_path, engine="openpyxl") as writer:
        for sheet_name, df in non_empty.items():
            pdf = df.toPandas()
            pdf.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            total_linhas += len(pdf)

    dbutils.fs.mkdirs(folder_path)
    dbutils.fs.cp(f"file:{local_tmp_path}", dbfs_path, True)
    os.remove(local_tmp_path)

    execucao_steps.append({
        "etapa": "Exportacao QAQC (Excel)",
        "status": "Sucesso",
        "registros_lidos": total_linhas, "registros_escritos": total_linhas,
        "flags": "\u2014",
        "observacoes": f"Excel gravado em {dbfs_path} -- abas: {', '.join(non_empty.keys())}",
    })
    return dbfs_path, filename


folder_silver_qaqc = project_path + "/SILVER/qaqc_continuo"

sheets_qaqc = {
    "Surrogate": df_api.filter(col("flag_surrogate").isNotNull()),
    "Totais_Dissolvidos": df_api.filter(col("flag_totais_dissolvido").isNotNull()),
    "DBO_DQO": df_api.filter(col("flag_dbo_dqo").isNotNull()),
    "Cromo": df_api.filter(col("flag_cromo").isNotNull()),
    "Branco": df_api.filter(col("flag_branco").isNotNull()),
    "RPD_Replica": df_api.filter(col("flag_rpd").isNotNull()),
}

resultado_export = export_qaqc_excel(sheets_qaqc, folder_silver_qaqc)

## Envio para o SharePoint

In [ ]:
if resultado_export is not None:
    dbfs_path, filename = resultado_export
    try:
        local_path = f"/tmp/{filename}"
        dbutils.fs.cp(dbfs_path, f"file:{local_path}")

        result = upload_file(
            folder_path=cfg["sharepoint_qaqc_folder"],
            local_path=local_path,
            filename=filename,
        )

        execucao_steps.append({
            "etapa": "Envio SharePoint (QAQC)",
            "status": "Sucesso" if result.get("status") == "success" else "Erro",
            "registros_lidos": 1, "registros_escritos": 1, "flags": "\u2014",
            "observacoes": f"Enviado '{filename}' para {cfg['sharepoint_qaqc_folder']}",
        })
        os.remove(local_path)
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Envio SharePoint (QAQC)", "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
            "observacoes": str(_e),
        })
else:
    print("Nada pra enviar -- nenhuma checagem de QAQC gerou linha flagada nesta execucao.")

display(pd.DataFrame(execucao_steps))
persistir_log()